# 04 — Vertical Spreads Lab: Four Ways to Trade a Direction

You will build all four verticals, summarize their defined risk, overlay the **debit vs credit**
expressions of the *same* bullish view, and trace the **POP vs max-profit** frontier across short
strikes.

DEMO: spot **$100**, IV **0.25**, **45 DTE**. Chain mids from module 00.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, viz


In [ ]:
SPOT, VOL, t = 100.0, 0.25, 45/365

def show(pos):
    s = analyzer.summarize(pos, SPOT, VOL)
    print(s['label'])
    print(f"  net_premium {s['net_premium']:+.0f}  breakevens {[round(b,2) for b in s['breakevens']]}")
    print(f"  max_profit {s['max_profit']:.0f}  max_loss {s['max_loss']:.0f}  POP {s['probability_of_profit']:.2f}")

## 1. The two debit verticals (bullish / bearish, low-IV expressions)

In [ ]:
bull_call = strategies.bull_call_spread((100, 3.91), (110, 0.73), expiry=t)
bear_put  = strategies.bear_put_spread((100, 3.42), (90, 0.62), expiry=t)
show(bull_call); print(); show(bear_put)

Bull call: risk **318** to make **682**, breakeven 103.18. Max profit + max loss = the $1000
width. Both are debit trades — you paid for net optionality (small long vega).

## 2. The two credit verticals (bullish / bearish, high-IV expressions)

In [ ]:
bull_put  = strategies.bull_put_spread((95, 1.58), (90, 0.62), expiry=t)
bear_call = strategies.bear_call_spread((105, 1.85), (110, 0.73), expiry=t)
show(bull_put); print(); show(bear_call)

Bull put: collect **96**, keep it if DEMO stays above 95; risk **404**. Note the **higher POP**
than the debit spreads — you profit even if the stock just sits still or drifts down a little.

## 3. Same view, two expressions: debit vs credit

The bull call spread and the bull put spread are **both bullish**. Overlay their expiry payoffs.
The debit version needs a move up; the credit version wins on 'not down'.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_compare([bull_call, bull_put], ax=ax)
ax.set_title('Two bullish expressions: bull call (debit) vs bull put (credit)')
plt.show()

Which to pick is an **IV decision**: debit (long vega) in low IV, credit (short vega) in high IV.
Confirm the vega signs from the summaries.

In [ ]:
gc = analyzer.summarize(bull_call, SPOT, VOL)['greeks']
gp = analyzer.summarize(bull_put,  SPOT, VOL)['greeks']
print(f'bull CALL spread  delta {gc.delta:+.1f}  vega {gc.vega:+.2f}  theta {gc.theta:+.2f}')
print(f'bull PUT  spread  delta {gp.delta:+.1f}  vega {gp.vega:+.2f}  theta {gp.theta:+.2f}')

## 4. The 50%-profit management rule (credit spread)

Credit spreads are managed at **50% of max profit**. Use `payoff.pnl_at` to see the mark-to-model
P&L of the bull put spread after time passes with the stock unchanged — decay pulls it toward the
50% target.

In [ ]:
from optionslab import payoff
for days in [0, 10, 20, 30]:
    pnl = payoff.pnl_at(bull_put, SPOT, days/365, VOL)
    print(f'{days:2d} days elapsed, spot flat at 100:  P&L {pnl:+.0f}  (max profit is 96)')

## 5. POP vs max-profit frontier

Slide the short put strike and watch the trade-off: closer strikes pay more but win less often.
You cannot maximize both — pick your point on the frontier.

In [ ]:
# (short_strike, short_prem, long_strike, long_prem) from the DEMO 45-DTE chain
variants = [(97.5, 2.37, 92.5, 1.01), (95, 1.58, 90, 0.62), (92.5, 1.01, 87.5, 0.37), (90, 0.62, 85, 0.22)]
for ks, ps, kl, pl in variants:
    sp = strategies.bull_put_spread((ks, ps), (kl, pl), expiry=t)
    s = analyzer.summarize(sp, SPOT, VOL)
    print(f'short {ks:5.1f}  credit {-s["net_premium"]/100:4.2f}  POP {s["probability_of_profit"]:.2f}  max_profit {s["max_profit"]:.0f}  max_loss {s["max_loss"]:.0f}')

Read down the columns: as the short strike moves further OTM (97.5 to 90), **POP rises** while the
**credit / max profit shrinks**. The ~30-delta strike (around 95) is the common balance point.

## 6. Payoff picture of a credit spread

`viz.plot_payoff` with the current spot and vol marks the profit zone and breakeven of the bull
put spread.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
viz.plot_payoff(bull_put, spot=SPOT, vol=VOL, ax=ax)
ax.set_title('Bull put spread — profit above 94.04, defined risk below')
plt.show()

## Experiments

1. In section 1, widen the bull call spread to **100/115** (short 115 call ~0.24). How do max
   profit, max loss, and breakeven change with the wider width?
2. In section 3, add the **long 100 call** alone to the `plot_compare`. How does the naked call's
   payoff differ from the two spreads (unlimited vs capped)?
3. In section 4, redo the P&L-over-time table with the stock drifting **down to 96** instead of
   flat. Does time decay still help enough, or does delta hurt more?
4. In section 5, build the analogous **bear call** frontier (slide the short call from 102.5 up to
   110). Is the credit at each delta larger or smaller than the put side, and why (skew)?
5. Re-summarize the bull put spread at `vol=0.40`. How much more credit would you collect, and why
   does that make credit spreads a *high-IV* strategy?